# Part 10 — Summary

Pure-CPU merger. Reads every `results_partN/summary.csv` written by Parts 01,
01b, 02, 04b, 05–09, and 11 and renders all aggregate tables inline. Writes a
single `final_results/summary.md` for hand-off.

Table 1 is the concat of three rows: Part 1 (rule-based) + Part 1b
(Llama Guard 2) + Part 11 (Llama Guard 3).

**Tolerant of missing artifacts** — if a Part hasn't run yet, the corresponding
section prints "Part N: not yet run" and the rest of the summary still renders.

Run this **last**, after every other Part has completed in its own kernel.


In [ ]:
import os
import pandas as pd
from pathlib import Path

RESULTS_DIRS = {
    "Part 1 (Rule-based judge)":          "results_part1",
    "Part 1b (Llama Guard 2 judge)":      "results_part1b",
    "Part 11 (Llama Guard 3 judge)":      "results_part11",
    "Part 2 (FP16 baseline refusal)":     "results_part2",
    "Part 4b (Perplexity sanity check)":  "results_part4b",
    "Part 5 (Table 2 attacks)":           "results_part5",
    "Part 6 (Table 3 defenses)":          "results_part6",
    "Part 7 (Figure 2 benign refusal)":   "results_part7",
    "Part 8 (Decoding sweep)":            "results_part8",
    "Part 9 (Cross-model)":               "results_part9",
}

def _safe_read(folder, **read_kwargs):
    path = Path(folder) / "summary.csv"
    if not path.exists():
        return None
    return pd.read_csv(path, **read_kwargs)

dfs = {label: _safe_read(folder) for label, folder in RESULTS_DIRS.items()}

# Part 6 was written with the defense names as a row index (no index=False),
# so re-read it preserving that.
if dfs["Part 6 (Table 3 defenses)"] is None:
    path = Path("results_part6") / "summary.csv"
    if path.exists():
        dfs["Part 6 (Table 3 defenses)"] = pd.read_csv(path, index_col=0)


In [ ]:
print("=" * 72)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("=" * 72)

# --- A: Table 1 (Part 1 rule-based + Part 1b Guard-2 + Part 11 Guard-3) ---
table1_parts = []
for key in ("Part 1 (Rule-based judge)",
            "Part 1b (Llama Guard 2 judge)",
            "Part 11 (Llama Guard 3 judge)"):
    if dfs[key] is not None:
        table1_parts.append(dfs[key])
if table1_parts:
    table1 = pd.concat(table1_parts, ignore_index=True)
    print("\nA. Table 1 — Judge Selection")
    print(table1.to_string(index=False))
else:
    print("\nA. Table 1 — Parts 1, 1b, 11 not yet run")

# --- B: Part 2 (FP16 refusal) ---
if dfs["Part 2 (FP16 baseline refusal)"] is not None:
    print("\nB. FP16 Baseline Refusal Rate on Harmful Goals (Part 2)")
    print(dfs["Part 2 (FP16 baseline refusal)"].to_string(index=False))
else:
    print("\nB. Part 2 not yet run")

# --- C: Part 4b (PPL) ---
if dfs["Part 4b (Perplexity sanity check)"] is not None:
    print("\nC. Perplexity (PPL) Sanity Check (Part 4b)")
    print(dfs["Part 4b (Perplexity sanity check)"].to_string(index=False))
else:
    print("\nC. Part 4b not yet run")

# --- D: Table 2 (Part 5 attacks, Guard-3 judge) ---
if dfs["Part 5 (Table 2 attacks)"] is not None:
    print("\nD. Table 2 — Attack ASR on Llama-2-7B-chat (Llama-Guard-3-8B judge)")
    print(dfs["Part 5 (Table 2 attacks)"].to_string(index=False))
else:
    print("\nD. Part 5 not yet run")

# --- E: Table 3 (Part 6 defenses) ---
if dfs["Part 6 (Table 3 defenses)"] is not None:
    print("\nE. Table 3 — Transfer-Attack ASR Under Defenses (Llama-Guard-3-8B judge)")
    print(dfs["Part 6 (Table 3 defenses)"].to_string())
else:
    print("\nE. Part 6 not yet run")

# --- F: Figure 2 (Part 7 benign refusal) ---
if dfs["Part 7 (Figure 2 benign refusal)"] is not None:
    print("\nF. Figure 2 — Refusal Rate on Benign Behaviors (keyword judge)")
    print(dfs["Part 7 (Figure 2 benign refusal)"].to_string(index=False))
else:
    print("\nF. Part 7 not yet run")

# --- G: Decoding sweep (Part 8) ---
if dfs["Part 8 (Decoding sweep)"] is not None:
    print("\nG. Decoding Parameter Sweep — ASR (Llama-Guard-3-8B judge)")
    print(dfs["Part 8 (Decoding sweep)"].to_string(index=False))
else:
    print("\nG. Part 8 not yet run")

# --- H: Cross-model (Part 9) ---
if dfs["Part 9 (Cross-model)"] is not None:
    print("\nH. Cross-Model Comparison (Llama-Guard-3-8B judge for PAIR ASR)")
    print(dfs["Part 9 (Cross-model)"].to_string(index=False))
else:
    print("\nH. Part 9 not yet run")


In [ ]:
# Write a markdown summary file for handoff.
os.makedirs("final_results", exist_ok=True)
lines = ["# JailbreakBench Reproduction — Comprehensive Summary", ""]

def _add(title, df, index=False):
    lines.append(f"## {title}")
    lines.append("")
    if df is None:
        lines.append("_Not yet run._")
    else:
        lines.append(df.to_markdown(index=index))
    lines.append("")

# A. Table 1
if table1_parts:
    _add("A. Table 1 — Judge Selection", table1, index=False)
else:
    _add("A. Table 1 — Judge Selection", None)

_add("B. FP16 Baseline Refusal Rate (Part 2)",
     dfs["Part 2 (FP16 baseline refusal)"])
_add("C. Perplexity Sanity Check (Part 4b)",
     dfs["Part 4b (Perplexity sanity check)"])
_add("D. Table 2 — Attack ASR (Part 5, Llama-Guard-3-8B judge)",
     dfs["Part 5 (Table 2 attacks)"])
_add("E. Table 3 — Defense ASR (Part 6)",
     dfs["Part 6 (Table 3 defenses)"], index=True)
_add("F. Figure 2 — Benign Refusal (Part 7)",
     dfs["Part 7 (Figure 2 benign refusal)"])
_add("G. Decoding Sweep (Part 8)",
     dfs["Part 8 (Decoding sweep)"])
_add("H. Cross-Model (Part 9)",
     dfs["Part 9 (Cross-model)"])

with open("final_results/summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Wrote final_results/summary.md")
